# significance-test

In [ ]:
from pathlib import Path

import seaborn as sns
from bonner.plotting import save_figure
from matplotlib import pyplot as plt
from matplotlib.ticker import LogFormatterMathtext
from tqdm.auto import tqdm

from lib.datasets import (
    compute_shared_stimuli,
    filter_by_stimulus,
    nsd,
    split_by_repetition,
)
from lib.spectra import (
    compute_cross_individual_spectra,
    compute_spectra_with_n_fold_cross_validation,
    compute_within_individual_spectra,
    plot_spectrum,
)
from lib.utilities import (
    JOURNAL_MATPLOTLIBRC,
    compute_best_fit,
    extract_geometrically_spaced_bins,
    mathtext_exponent_label,
)

FIGURES_HOME = Path.cwd().parent / "figures"
FIGURES_HOME.mkdir(exist_ok=True, parents=True)

sns.set_theme(context="paper", style="ticks", rc=JOURNAL_MATPLOTLIBRC)

REFERENCE_SUBJECT = 0
REFERENCE_SUBJECT_COMPARISON = 1

## load datasets

In [ ]:
datasets = {
    roi: {
        subject: nsd.load_dataset(
            subject=subject,
            roi=roi,
            preprocessing="fithrf",
            z_score=True,
        )
        for subject in tqdm(range(nsd.N_SUBJECTS), desc="subject", leave=False)
    }
    for roi in tqdm(("general", "frontal"), desc="region of interest", leave=False)
}

datasets_within = {
    roi: {
        subject: split_by_repetition(
            filter_by_stimulus(
                dataset,
                stimuli=compute_shared_stimuli([dataset], n_repetitions=2),
            ),
            n_repetitions=2,
        )
        for subject, dataset in tqdm(datasets_.items(), desc="subject", leave=False)
    }
    for roi, datasets_ in tqdm(datasets.items(), desc="region of interest", leave=False)
}

shared_stimuli = compute_shared_stimuli(datasets["general"].values(), n_repetitions=2)

datasets_within = {
    roi: {
        subject: datasets_[subject]
        for subject in (REFERENCE_SUBJECT, REFERENCE_SUBJECT_COMPARISON)
    }
    for roi, datasets_ in datasets_within.items()
}

datasets_cross = {
    roi: {
        subject: split_by_repetition(
            filter_by_stimulus(datasets_[subject], stimuli=shared_stimuli),
            n_repetitions=2,
        )
        for subject in tqdm(
            (REFERENCE_SUBJECT, REFERENCE_SUBJECT_COMPARISON),
            desc="subject",
            leave=False,
        )
    }
    for roi, datasets_ in tqdm(datasets.items(), desc="region of interest", leave=False)
}

## compute spectra

In [ ]:
spectra_within = {
    roi: compute_within_individual_spectra(
        datasets_within[roi],
        n_permutations=5_000,
    )
    for roi in ("general", "frontal")
}

spectra_cross = {
    roi: compute_cross_individual_spectra(
        datasets_cross[roi],
        reference_individual=REFERENCE_SUBJECT,
        n_permutations=5_000,
        randomized=True,
    )
    for roi in ("general", "frontal")
}

dataset = datasets_within["general"][REFERENCE_SUBJECT]

nulls_samples = (
    compute_spectra_with_n_fold_cross_validation(
        x_train=dataset[0],
        y_train=dataset[1],
        x_test=dataset[0],
        y_test=dataset[1],
        n_folds=8,
        n_permutations=5_000,
    )["covariance (permuted)"]
    .isel(component=range(10))
    .mean("fold")
)

## plot significance-test

In [ ]:
fig, axes = plt.subplots(figsize=(5.5, 4.5), ncols=2, sharey=True, sharex=True)

kwargs_legend = {
    "loc": "upper right",
    "title": "subject",
    "ncols": 2,
    "columnspacing": 0.5,
    "handletextpad": 0.25,
}

kwargs = {
    "general": {
        "within": {
            "c": sns.color_palette("crest_r", n_colors=8)[REFERENCE_SUBJECT],
        },
        "cross": {
            "c": sns.color_palette("flare_r", n_colors=8)[1],
        },
    },
    "frontal": {
        "c": "gray",
    },
}

null_quantiles = (0.997, 0.955, 0.683)
palette = sns.color_palette("Grays", n_colors=len(null_quantiles))[::-1]
palette_text = sns.color_palette("Grays", n_colors=5)[::-1][: len(null_quantiles)]

ax = axes[0]
plot_spectrum(
    ax=ax,
    spectrum=spectra_within["general"].sel(individual=REFERENCE_SUBJECT),
    ls="None",
    marker="s",
    hide_insignificant=True,
    null_quantile=0.999,
    **kwargs["general"]["within"],
)
ax.set_ylabel("covariance")
ax.set_xlabel("rank")
ax.set_title(
    f"within-subject,\nsubject {REFERENCE_SUBJECT + 1}",
    pad=20,
)

# compute slope of spectrum
spectrum = spectra_within["general"]["covariance"].sel(individual=REFERENCE_SUBJECT)
spectrum_mean = spectrum.mean("fold")
ranks = list(range(spectrum_mean.sizes["rank"] - 2))
spectrum_mean = spectrum_mean.isel(rank=ranks)
x = spectrum_mean["rank"].to_numpy()
func, slope = compute_best_fit(
    x,
    spectrum_mean.to_numpy(),
)
bin_edges, bin_centers = extract_geometrically_spaced_bins(
    start=1,
    stop=10_000,
    density=3,
)
edges = bin_edges[:10]
y_predicted = func(edges)
ax.plot(edges, y_predicted, ls="--", c="lightgray")
print(f"power-law exponent for within-individual spectrum: {slope:.2f}")

y_predicted = func(bin_centers[:9])
spectrum = spectrum.isel({"rank": list(range(len(y_predicted)))})
chi_squared = (
    (((spectrum.mean("fold") - y_predicted) ** 2) / spectrum.var("fold")).sum("rank")
) / (spectrum.sizes["rank"] - 2)
print(f"reduced chi-squared = {chi_squared:.2f}")

positions = [
    (5e1, 1e-5),
    (1e2, 1.5e-6),
    (2e2, 2e-7),
]
for quantile, color, textcolor, position in zip(
    null_quantiles,
    palette,
    palette_text,
    positions,
    strict=True,
):
    nulls = (
        spectra_within["general"]["covariance (permuted)"]
        .mean("fold")
        .quantile(quantile, dim="permutation")
        .sel(individual=REFERENCE_SUBJECT)
    )
    null_line = ax.plot(
        nulls["rank"].to_numpy(),
        nulls.to_numpy(),
        ls="-",
        c=color,
        mew=0,
        alpha=0.75,
    )
    ax.text(
        position[0],
        position[1],
        f"{quantile * 100:.1f}%",
        c=textcolor,
        ha="center",
        va="center",
        fontsize="xx-small",
        backgroundcolor="white",
        bbox={"facecolor": "white", "alpha": 0.75, "pad": 1},
    )

with plt.rc_context(
    rc=JOURNAL_MATPLOTLIBRC
    | {
        "axes.linewidth": 0.75,
        "xtick.major.size": 0,
        "ytick.major.size": 0,
        "xtick.labelsize": "x-small",
        "ytick.labelsize": "x-small",
        "axes.titlesize": "x-small",
    },
):
    bottom = 0.35
    ax = axes[0].inset_axes([0.1, 0.1, 0.36, 0.3])
    sns.lineplot(
        ax=ax,
        data=nulls_samples.isel(permutation=range(30)).to_dataframe().reset_index(),
        x="component",
        y="covariance (permuted)",
        color="darkgray",
        estimator=None,
        units="permutation",
        lw=0.5,
        alpha=0.5,
    )
    sns.lineplot(
        ax=ax,
        data=nulls_samples.to_dataframe().reset_index(),
        x="component",
        y="covariance (permuted)",
        color="dimgray",
        estimator="mean",
        errorbar="se",
        err_style="bars",
        ls="None",
        marker="o",
        mew=0,
        ms=3,
    )
    ax.set_ylabel("")
    ax.set_xlabel("")
    ax.axhline(0, ls="-", c="gray", lw=1)
    ax.set_xticks([1, 10])
    ax.set_ylim(bottom=-1e-3, top=1e-3)
    ax.set_xlim(left=1e0, right=1e1)
    ax.xaxis.set_major_formatter(LogFormatterMathtext())
    ax.ticklabel_format(axis="y", scilimits=(-4, -4))
    ax.set_yticks([-8e-4, 0, 8e-4])
    ax.text(
        6.5,
        -7.5e-4,
        "with randomly\nshuffled images",
        ha="center",
        va="baseline",
        fontsize="xx-small",
    )

ax_inset = axes[0].inset_axes(
    (0.5, 0.85, 0.5, 0.5),
    xlim=(1, 1.5e2),
    ylim=(1e-5, 1e-1),
    xticklabels=[],
    yticklabels=[],
)
for roi in ("general", "frontal"):
    plot_spectrum(
        ax=ax_inset,
        spectrum=spectra_within[roi].sel(individual=REFERENCE_SUBJECT),
        ls="None",
        marker="s",
        label="visual" if roi == "general" else "frontal",
        hide_insignificant=True,
        null_quantile=0.999,
        **kwargs["general"]["within"] if roi == "general" else kwargs["frontal"],
    )
    ax_inset.set_xscale("log")
    ax_inset.set_yscale("log")
    ax_inset.legend(
        loc="upper right",
        fontsize="small",
        title_fontsize="small",
        handletextpad=0.5,
        borderaxespad=0,
        borderpad=0,
    )
    ytick_exponents = list(range(-5, 0))
    ax_inset.set_yticks(
        [10**exponent for exponent in ytick_exponents],
        labels=[
            mathtext_exponent_label(exponent) if exponent % 2 == 1 else ""
            for exponent in ytick_exponents
        ],
    )

ax = axes[1]
plot_spectrum(
    ax=ax,
    spectrum=spectra_cross["general"].sel(individual=REFERENCE_SUBJECT_COMPARISON),
    ls="None",
    marker="o",
    hide_insignificant=True,
    null_quantile=0.999,
    **kwargs["general"]["cross"],
)
ax.set_title(
    (
        f"between-subject,\nsubject {REFERENCE_SUBJECT_COMPARISON + 1}"
        f" relative to subject {REFERENCE_SUBJECT + 1}"
    ),
    pad=20,
)
ax.yaxis.set_tick_params(labelbottom=True)
ax.set_ylabel("cross-covariance")
ax.set_xlabel("rank")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(left=1, right=1e4)
ax.set_xticks([1, 1e1, 1e2, 1e3, 1e4])
ax.set_ylim(top=1e-1, bottom=1e-8)
ytick_exponents = list(range(-8, 0))
ax.set_yticks(
    [10**exponent for exponent in ytick_exponents],
    labels=[
        mathtext_exponent_label(exponent) if exponent % 2 == 1 else ""
        for exponent in ytick_exponents
    ],
)
ax.axvline(len(shared_stimuli), ls="--", c="gray", lw=0.5, ymax=0.45)
ax.text(
    s="number of\nshared images",
    x=1e3,
    y=3e-5,
    fontsize="xx-small",
    ha="center",
    va="bottom",
)

spectrum = spectra_cross["general"]["covariance"].sel(
    individual=REFERENCE_SUBJECT_COMPARISON
)
spectrum_mean = spectrum.mean("fold")
ranks = list(range(spectrum_mean.sizes["rank"] - 3))
spectrum_mean = spectrum_mean.isel(rank=ranks)
x = spectrum_mean["rank"].to_numpy()
func, slope = compute_best_fit(
    x,
    spectrum_mean.to_numpy(),
)
edges = bin_edges[:7]
y_predicted = func(edges)
ax.plot(edges, y_predicted, ls="--", c="lightgray")
print(f"power-law exponent for cross-individual spectrum: {slope:.2f}")

y_predicted = func(bin_centers[:6])
spectrum = spectrum.isel({"rank": list(range(len(y_predicted)))})
chi_squared = (
    (((spectrum.mean("fold") - y_predicted) ** 2) / spectrum.var("fold")).sum("rank")
) / (spectrum.sizes["rank"] - 2)
print(f"reduced chi-squared = {chi_squared:.2f}")

positions = [
    (8e0, 2e-4),
    (1.5e1, 4e-5),
    (2.5e1, 8e-6),
]
for quantile, color, textcolor, position in zip(
    null_quantiles,
    palette,
    palette_text,
    positions,
    strict=True,
):
    nulls = (
        spectra_cross["general"]["covariance (permuted)"]
        .mean("fold")
        .quantile(quantile, dim="permutation")
        .sel(individual=REFERENCE_SUBJECT_COMPARISON)
    )
    null_line = ax.plot(
        nulls["rank"],
        nulls,
        ls="-",
        c=color,
        mew=0,
        alpha=0.75,
    )
    ax.text(
        position[0],
        position[1],
        f"{quantile * 100:.1f}%",
        c=textcolor,
        ha="center",
        va="center",
        fontsize="xx-small",
        backgroundcolor="white",
        bbox={"facecolor": "white", "alpha": 0.75, "pad": 1},
    )
ax.text(
    1e1,
    8e-7,
    "percentiles of\nnull distributtion",
    ha="center",
    va="center",
    fontsize="xx-small",
)

ax_inset = axes[1].inset_axes(
    (0.5, 0.85, 0.5, 0.5),
    xlim=(1, 1.5e2),
    ylim=(1e-5, 1e-1),
    xticklabels=[],
    yticklabels=[],
)
for roi in ("general", "frontal"):
    plot_spectrum(
        ax=ax_inset,
        spectrum=spectra_cross[roi].sel(individual=REFERENCE_SUBJECT_COMPARISON),
        ls="None",
        marker="o",
        label="visual" if roi == "general" else "frontal",
        hide_insignificant=True,
        null_quantile=0.999,
        **kwargs["general"]["cross"] if roi == "general" else kwargs["frontal"],
    )
    ax_inset.set_xscale("log")
    ax_inset.set_yscale("log")
    ax_inset.legend(
        loc="upper right",
        fontsize="small",
        title_fontsize="small",
        handletextpad=0.5,
        borderaxespad=0,
        borderpad=0,
    )
    ytick_exponents = list(range(-5, 0))
    ax_inset.set_yticks(
        [10**exponent for exponent in ytick_exponents],
        labels=[
            mathtext_exponent_label(exponent) if exponent % 2 == 1 else ""
            for exponent in ytick_exponents
        ],
    )

save_figure(fig, filepath=FIGURES_HOME / "significance-test.pdf")